# Tutorial 01 — Defining a soft body assembly

A *soft body* in SoftMobility is a collection of rigid spheres whose geometry depends on three groups of variables:

| Group | What it is | Example |
|---|---|---|
| **Degrees of freedom (DOFs)** | Internal variables integrated during simulation | bending angle, extension |
| **Design parameters** | Fixed geometric / material properties; targets for optimisation | sphere radii, spring stiffness |
| **Inputs** | External signals provided at run time by the solver | gravity, magnetic field |

This tutorial shows two equivalent ways to define a body:
1. **YAML format** — compact, readable, and the recommended approach
2. **Python callables** — for geometry that requires arbitrary Python code

*Prerequisites: none.  Next: Tutorial 02 (flows and inputs), Tutorial 04 (Jeffery orbits).*

In [1]:
import os
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

import softmobility as sm
from softmobility import SoftBody, SphereAssembly, Sphere

## 1 — YAML format

A YAML string (or file path) is parsed by `SoftBody` into a fully JAX-compatible assembly.  The description has four sections — only `spheres` is required:

```yaml
dof_names:     # registers variable prefixes for DOFs       (optional)
design_names:  # registers variable prefixes for design     (optional)
defaults:      # numeric default values                     (optional)
spheres:       # list of sphere descriptions                (required)
```

Every expression you write in a sphere field is compiled through SymPy into a JAX function, so the assembly can be differentiated and JIT-compiled without any extra work.

### Step 1 — two rigid spheres (minimal example)

The simplest possible body: two unit spheres at fixed positions, no degrees of freedom.  This is the classic symmetric dumbbell used in Tutorial 04 (Jeffery orbits).

In [2]:
yaml_rigid = """
spheres:
  - radius: 1.0
    position: [-1.5, 0, 0]
  - radius: 1.0
    position: [ 1.5, 0, 0]
"""

body_rigid = SoftBody(yaml_rigid, verbose=False)
print(repr(body_rigid))

SPHERE ASSEMBLY
  2 spheres
  0 degrees of freedom
  0 design parameters
  0 input parameters

Default values
  degrees of freedom dof: [] = []
  design parameters param: [] = []
  input parameters param: []

SPHERE 0
  radius: 1.0
  position: [-1.5  0.   0. ]
  orientation: [0. 0. 0.]
  C_H:
[]
  C_K:
[]

SPHERE 1
  radius: 1.0
  position: [1.5 0.  0. ]
  orientation: [0. 0. 0.]
  C_H:
[]
  C_K:
[]



### Step 2 — adding a degree of freedom

A **DOF** is an internal variable that the solver integrates in time. Declare a prefix in `dof_names` and use indexed names (`x0`, `x1`, …) in sphere expressions.

```yaml
dof_names: [x]   # registers prefix "x" → variables x0, x1, x2, ... are DOFs
```

Default values come from the `defaults` section, keyed by the full variable name (`x0: 0.1`).  Variables absent from `defaults` are initialised to zero.

Here `x0` is the rotation angle (in radians) of each sphere around the body *y*-axis — a tilt DOF.  The checkered sphere visualisation in Part 3 makes this rotation visible.

In [3]:
yaml_dof = """
dof_names: [x]         # "x" prefix: x0, x1, ... are DOFs

defaults:
  x0: 0.1              # initial tilt angle (rad)

spheres:
  - radius: 1.0
    position: [-1.5, 0, 0]
    orientation: [0, x0, 0]    # Rodrigues vector: sphere 0 tilts by angle x0 around y

  - radius: 1.0
    position: [ 1.5, 0, 0]
    orientation: [0, -x0, 0]   # sphere 1 tilts opposite (antisymmetric)
"""

body_dof = SoftBody(yaml_dof, verbose=False)
print(repr(body_dof))
print()
print("DOF variables:", body_dof.dof_variables)   # ['x0']
print("DOF defaults: ", body_dof.dof_defaults)    # [0.1]

SPHERE ASSEMBLY
  2 spheres
  1 degrees of freedom
  0 design parameters
  0 input parameters

Default values
  degrees of freedom dof: ['x0'] = [0.1]
  design parameters param: [] = []
  input parameters param: []

SPHERE 0
  radius: 1.0
  position: [-1.5  0.   0. ]
  orientation: [0.  0.1 0. ]
  C_H:
[]
  C_K:
[[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]

SPHERE 1
  radius: 1.0
  position: [1.5 0.  0. ]
  orientation: [ 0.  -0.1  0. ]
  C_H:
[]
  C_K:
[[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]


DOF variables: ['x0']
DOF defaults:  [0.1]


### Step 3 — design parameters

**Design parameters** are fixed during a simulation but are the natural targets for gradient-based optimisation (Tutorial 06).  Declare prefixes in `design_names`.

> ⚠️ **Alphabetical ordering.**  The parser sorts all variable names alphabetically within each group.  `design_names: [radius, length, k]` produces `body.design_variables = ['k', 'length', 'radius']`.  Always check `body.design_variables` before constructing arrays.

In [4]:
yaml_design = """
dof_names:    [x]
design_names: [radius, length]

defaults:
  x0:     0.1
  radius: 0.5
  length: 2.0

spheres:
  - radius: radius
    position: [-length / 2, 0, 0]
    orientation: [0, x0, 0]

  - radius: radius
    position: [ length / 2, 0, 0]
    orientation: [0, -x0, 0]
"""

body_design = SoftBody(yaml_design, verbose=False)

# Note the alphabetical order — not the order listed in design_names
print("Design variables:", body_design.design_variables)   # ['length', 'radius']
print("Design defaults: ", body_design.design_defaults)    # [ 2.0,  0.5 ]

Design variables: ['length', 'radius']
Design defaults:  [2.  0.5]


### Step 4 — forces, torques, and spring coupling

`force` and `torque` entries define elastic restoring forces.  The parser extracts the *stiffness coupling matrix* $\mathbf{C}_K$ as the linear operator that relates the DOF vector $\boldsymbol{Q}$ and the six-component force vector (force+torque) $\boldsymbol{f}$:
$$\boldsymbol{f} = [\boldsymbol
{F},\,\boldsymbol{T}] = \mathbf{C}_K(\mathrm{dofs},\mathrm{design})\cdot\boldsymbol{Q}$$
Expressions must be **linear in DOFs**.

A torsion spring of stiffness $k$ restoring $x_0 \to 0$:

```
sphere 0:  torque_y = -k * x0   (restoring)
sphere 1:  torque_y = +k * x0   (equal-and-opposite reaction)
```

In [5]:
yaml_spring = """
dof_names:    [x]
design_names: [radius, length, k]

defaults:
  x0:     0.1
  radius: 0.5
  length: 2.0
  k:      1.0

spheres:
  - radius: radius
    position: [-length / 2, 0, 0]
    orientation: [0, x0, 0]
    torque: [0, -k * x0, 0]     # restoring torque on sphere 0

  - radius: radius
    position: [ length / 2, 0, 0]
    orientation: [0, -x0, 0]
    torque: [0, k * x0, 0]      # equal-and-opposite reaction on sphere 1
"""

body_spring = SoftBody(yaml_spring, verbose=False)
print(repr(body_spring))

SPHERE ASSEMBLY
  2 spheres
  1 degrees of freedom
  3 design parameters
  0 input parameters

Default values
  degrees of freedom dof: ['x0'] = [0.1]
  design parameters param: ['k', 'length', 'radius'] = [1.  2.  0.5]
  input parameters param: []

SPHERE 0
  radius: 0.5
  position: [-1.  0.  0.]
  orientation: [0.  0.1 0. ]
  C_H:
[]
  C_K:
[[ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [-1.]
 [ 0.]]

SPHERE 1
  radius: 0.5
  position: [1. 0. 0.]
  orientation: [ 0.  -0.1  0. ]
  C_H:
[]
  C_K:
[[0.]
 [0.]
 [0.]
 [0.]
 [1.]
 [0.]]



### Step 5 — inputs: fields and scalars

**Inputs** are time-varying external signals.  They may only appear in `force` and `torque` expressions, and must be **linear** in the input variables.

Two kinds of input:

| Kind | YAML names | How to provide at run time |
|---|---|---|
| **3-D field** | base name with suffixes `0`, `1`, `2` (e.g. `gravity0`, `gravity1`, `gravity2`) | `Field` object in `input_map` |
| **Scalar** | base name without suffix (e.g. `act_torque`) | `Scalar` object in `input_map` |

Register the base name in `input_names`.  All three components of a field must appear in at least one sphere expression.

The file `parameters.yaml` demonstrates both kinds.  Load it below.

In [6]:
yaml_path = os.path.join(os.path.dirname(sm.__file__), 'tutorials', 'parameters.yaml')

body_full = SoftBody(yaml_path, verbose=True)
print()
print("DOF variables:    ", body_full.dof_variables)
print("Design variables: ", body_full.design_variables)
print("Input variables:  ", body_full.input_variables)

Parsing parameter file: /Users/Ch/Documents/Python/SoftMobility/softmobility/tutorials/parameters.yaml
  Found variables: x0, x1, distance, myradius, spring_k, gravity0, gravity1, gravity2, act_torque
  3D field inputs:  ['gravity']
  Scalar inputs:    ['act_torque']
    Sphere 0
      Radius: myradius
      Position: ['0', '0', '0']
      Orientation: ['x0', 'x1', '0']
      Force: ['1.0*gravity0', '1.0*gravity1', '1.0*gravity2']
      Torque: ['act_torque - spring_k*x0', '-spring_k*x1', '0']
    Sphere 1
      Radius: 1 - myradius
      Position: ['0', '0', 'distance + 1']
      Orientation: ['-myradius*x0/(1 - myradius)', '-myradius*x1/(1 - myradius)', '0']
      Force: ['-1.0*gravity0', '-1.0*gravity1', '-1.0*gravity2']
      Torque: ['-act_torque + spring_k*x0', 'spring_k*x1', '0']

DOF variables:     ['x0', 'x1']
Design variables:  ['distance', 'myradius', 'spring_k']
Input variables:   ['gravity0', 'gravity1', 'gravity2', 'act_torque']


**Constants** — any name in `defaults` that is *not* declared in `dof_names`, `design_names`, or `input_names` — are evaluated once at parse time and substituted numerically into every expression that uses them.

## 2 — Python callables

For geometry that requires arbitrary Python / NumPy logic (look-up tables, conditional branches, external data), build the assembly directly from callable objects.

`Sphere` accepts:

| Argument | Signature | Notes |
|---|---|---|
| `radius` | `(dofs, design) → float` | |
| `position` | `(dofs, design, time) → array(3,)` | |
| `orientation` | `(dofs, design, time) → array(3,)` | |
| `force` | `(dofs, design) → array(3,)` | |
| `torque` | `(dofs, design) → array(3,)` | |

In [7]:
# Build the same 1-DOF symmetric dumbbell from Step 4 using callables
sa = SphereAssembly()

sa.add_dof(name='x0', default=0.1)
sa.add_design(name='length', default=2.0)
sa.add_design(name='radius', default=0.5)
sa.add_design(name='k', default=1.0)

i_x0     = sa.dof_variables.index('x0')
i_length = sa.design_variables.index('length')
i_radius = sa.design_variables.index('radius')
i_k      = sa.design_variables.index('k')

sa.add_sphere(Sphere(
    radius      = lambda dofs, design:          design[i_radius],
    position    = lambda dofs, design, time:    np.array([-design[i_length] / 2, 0.0, 0.0]),
    orientation = lambda dofs, design, time:    np.array([ 0.0, dofs[i_x0], 0.0]),
    torque      = lambda dofs, design, inputs:  np.array([0.0, -design[i_k] * dofs[i_x0], 0.0]),
))
sa.add_sphere(Sphere(
    radius      = lambda dofs, design:          design[i_radius],
    position    = lambda dofs, design, time:    np.array([ design[i_length] / 2, 0.0, 0.0]),
    orientation = lambda dofs, design, time:    np.array([ 0.0, -dofs[i_x0], 0.0]),
    torque      = lambda dofs, design, inputs:  np.array([0.0, design[i_k] * dofs[i_x0], 0.0]),
))

print(repr(sa))

NEW degrees of freedom
 ['x0'] 
with default values
 [0.1]
NEW design parameters
 ['length'] 
with default values
 [2.]
NEW design parameters
 ['length', 'radius'] 
with default values
 [2.  0.5]
NEW design parameters
 ['length', 'radius', 'k'] 
with default values
 [2.  0.5 1. ]
SPHERE ASSEMBLY
  2 spheres
  1 degrees of freedom
  3 design parameters
  0 input parameters

Default values
  degrees of freedom dof: ['x0'] = [0.1]
  design parameters param: ['length', 'radius', 'k'] = [2.  0.5 1. ]
  input parameters param: []

SPHERE 0
  radius: 0.5
  position: [-1.  0.  0.]
  orientation: [0.  0.1 0. ]
  C_H:
[]
  C_K:
[[ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [-1.]
 [ 0.]]

SPHERE 1
  radius: 0.5
  position: [1. 0. 0.]
  orientation: [ 0.  -0.1  0. ]
  C_H:
[]
  C_K:
[[0.]
 [0.]
 [0.]
 [0.]
 [1.]
 [0.]]



The `repr` output confirms that geometry matches the YAML version, but `C_K` is empty because no spring torques were specified.  For this reason, in simulation workflows always start from `SoftBody(yaml_string)` rather than building sphere-by-sphere.

## 3 — Visualising the assembly

The helpers below render any `SphereAssembly` (or `SoftBody`) as an interactive 3-D Plotly figure.  Each sphere is drawn with a checkered surface so that its Rodrigues orientation is visible: rotate the slider to see the checker pattern twist.

In [8]:
def rotate_points(points, axis_angle):
    """Rotate an array of points by a Rodrigues rotation vector."""
    points = np.array(points)
    theta  = np.linalg.norm(np.array(axis_angle))
    if theta < 1e-8:
        return points
    k = axis_angle / theta
    kx, ky, kz = k
    K = np.array([[0, -kz, ky], [kz, 0, -kx], [-ky, kx, 0]])
    R = np.eye(3) + np.sin(theta) * K + (1 - np.cos(theta)) * K @ K
    return points @ R.T

def create_sphere(radius=1, center=(0, 0, 0), orientation=(0, 0, 0),
                  resolution=32, checkered_scale=8):
    """Return (x, y, z, colors) for a checkered sphere surface."""
    u = np.linspace(0, 2 * np.pi, resolution)
    v = np.linspace(0, np.pi,     resolution)
    x = radius * np.outer(np.cos(u), np.sin(v))
    y = radius * np.outer(np.sin(u), np.sin(v))
    z = radius * np.outer(np.ones(resolution), np.cos(v))
    pts = np.column_stack([x.ravel(), y.ravel(), z.ravel()])
    rot = rotate_points(pts, orientation)
    x_r = rot[:, 0].reshape(x.shape) + center[0]
    y_r = rot[:, 1].reshape(y.shape) + center[1]
    z_r = rot[:, 2].reshape(z.shape) + center[2]
    cu  = (np.floor(np.linspace(0, resolution, resolution) / checkered_scale) % 2).astype(int)
    cv  = (np.floor(np.linspace(0, resolution, resolution) / checkered_scale) % 2).astype(int)
    return x_r, y_r, z_r, np.add.outer(cu, cv) % 2

def add_arrow(fig, start, direction, color, name):
    """Add a labelled 3-D arrow (line + cone) to a Plotly figure."""
    end        = start + direction
    cone_start = start + 0.75 * direction
    fig.add_trace(go.Scatter3d(
        x=[start[0], end[0]], y=[start[1], end[1]], z=[start[2], end[2]],
        mode="lines", line=dict(color=color, width=5), name=f"{name}-axis"))
    fig.add_trace(go.Cone(
        x=[cone_start[0]], y=[cone_start[1]], z=[cone_start[2]],
        u=[direction[0]], v=[direction[1]], w=[direction[2]],
        colorscale=[[0, color], [1, color]], showscale=False,
        sizemode="absolute", sizeref=0.333 * np.linalg.norm(direction)))

def update_figure(fig, assembly, dofs=None, design=None):
    """Redraw the assembly for the given dofs and design arrays."""
    dofs   = np.array(dofs,   dtype=float) if dofs   is not None else np.array(assembly.dof_defaults,    dtype=float)
    design = np.array(design, dtype=float) if design is not None else np.array(assembly.design_defaults, dtype=float)
    fig.data = []
    for axis, color, name in zip(np.eye(3), ['blue', 'red', 'green'], ['x', 'y', 'z']):
        add_arrow(fig, np.zeros(3), axis.astype(float), color, name)
    for i, sphere in enumerate(assembly.spheres):
        x, y, z, colors = create_sphere(
            radius      = sphere.radius(dofs, design),
            center      = sphere.position(dofs, design, [0.0]),
            orientation = sphere.orientation(dofs, design, [0.0]),
        )
        fig.add_trace(go.Surface(
            x=x, y=y, z=z, surfacecolor=colors,
            showscale=False, cmin=0,
            cmax=assembly.Nspheres / (1 + i),
            colorscale='Thermal',
        ))
    fig.update_layout(
        title="Soft body — body frame",
        autosize=False, width=700, height=700,
        scene=dict(xaxis_visible=False, yaxis_visible=False, zaxis_visible=False),
        scene_camera=dict(projection=dict(type="orthographic")),
    )

### Interactive viewer

Two sliders let you explore the body interactively:

- **x0 (DOF)**: the tilt angle of each sphere around the body *y*-axis.    Watch the checkered pattern rotate — a non-zero Rodrigues vector is hard   to see from positions alone, but the checker makes it obvious.
- **length**: the centre-to-centre separation (a design parameter).

In [9]:
body = SoftBody(yaml_spring, verbose=False)

# Retrieve variable indices (do not assume a fixed order — always look up)
i_x0     = body.dof_variables.index('x0')
i_length = body.design_variables.index('length')

x0_slider     = widgets.FloatSlider(min=-0.8, max=0.8, step=0.05, value=0.1, description='x0  (DOF)')
length_slider = widgets.FloatSlider(min=0.5,  max=4.0, step=0.1,  value=2.0, description='length')

fig = go.Figure()

def update(x0, length):
    dofs   = np.array(body.dof_defaults,    dtype=float).copy()
    design = np.array(body.design_defaults, dtype=float).copy()
    dofs[i_x0]       = x0
    design[i_length] = length
    update_figure(fig, body, dofs=dofs, design=design)
    fig.show()

widgets.interactive(update, x0=x0_slider, length=length_slider)

interactive(children=(FloatSlider(value=0.1, description='x0  (DOF)', max=0.8, min=-0.8, step=0.05), FloatSlid…